In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error,mean_squared_error

from tensorflow.keras.models import Sequential,load_model
from tensorflow.keras.layers import SimpleRNN, Dense,Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [4]:
df = pd.read_csv('venv/household_power_consumption.csv', low_memory=False)
print('\n Dataset loaded successfully')


 Dataset loaded successfully


In [5]:
print('\n Shape of the dataset')
print(df.shape)
print('First 5 rows ')
print(df.head())
print('\n Dataset information')
print(df.info())
print('\n Missing Values')
print(df.isnull().sum())


 Shape of the dataset
(1048575, 9)
First 5 rows 
         Date      Time Global_active_power Global_reactive_power Voltage  \
0  16/12/2006  17:24:00               4.216                 0.418  234.84   
1  16/12/2006  17:25:00                5.36                 0.436  233.63   
2  16/12/2006  17:26:00               5.374                 0.498  233.29   
3  16/12/2006  17:27:00               5.388                 0.502  233.74   
4  16/12/2006  17:28:00               3.666                 0.528  235.68   

  Global_intensity Sub_metering_1 Sub_metering_2  Sub_metering_3  
0             18.4              0              1            17.0  
1               23              0              1            16.0  
2               23              0              2            17.0  
3               23              0              1            17.0  
4             15.8              0              1            17.0  

 Dataset information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entri

In [11]:
print(df.columns)

Index(['Date', 'Time', 'Global_active_power', 'Global_reactive_power',
       'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2',
       'Sub_metering_3'],
      dtype='object')


In [12]:
df.replace("?",np.nan,inplace=True)
numeric_columns=["Global_active_power","Global_reactive_power","Voltage","Global_intensity","Sub_metering_1","Sub_metering_2","Sub_metering_3"]

for col in numeric_columns:
    df[col]=pd.to_numeric(df[col],errors="coerce")
    
df.fillna(method="ffill",inplace=True)
df["Datatime"]=pd.to_datetime(df["Date"]+" "+df["Time"],dayfirst=True)

/var/folders/wp/hft_4m3d2nx9j8fsn18yl0f40000gn/T/ipykernel_62973/4243242787.py:7: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method="ffill",inplace=True)


In [16]:
df['Datetime'] = pd.to_datetime(
    df['Date'] + ' ' + df['Time'],
    dayfirst=True
)

In [17]:
df = df.sort_values("Datetime")

In [18]:
df.set_index("Datetime",inplace=True)

print("\nData After Cleaning")
print(df.head())


Data After Cleaning
                           Date      Time  Global_active_power  \
Datetime                                                         
2006-12-16 17:24:00  16/12/2006  17:24:00                4.216   
2006-12-16 17:25:00  16/12/2006  17:25:00                5.360   
2006-12-16 17:26:00  16/12/2006  17:26:00                5.374   
2006-12-16 17:27:00  16/12/2006  17:27:00                5.388   
2006-12-16 17:28:00  16/12/2006  17:28:00                3.666   

                     Global_reactive_power  Voltage  Global_intensity  \
Datetime                                                                
2006-12-16 17:24:00                  0.418   234.84              18.4   
2006-12-16 17:25:00                  0.436   233.63              23.0   
2006-12-16 17:26:00                  0.498   233.29              23.0   
2006-12-16 17:27:00                  0.502   233.74              23.0   
2006-12-16 17:28:00                  0.528   235.68              15.8   

    

In [19]:
features =[
    'Global_active_power',
    'Global_reactive_power',
    'Voltage',
    'Sub_metering_1',
    'Sub_metering_2',
    'Sub_metering_3'
]
data = df[features]
print("\nSelected Features")
print(data.head())


Selected Features
                     Global_active_power  Global_reactive_power  Voltage  \
Datetime                                                                   
2006-12-16 17:24:00                4.216                  0.418   234.84   
2006-12-16 17:25:00                5.360                  0.436   233.63   
2006-12-16 17:26:00                5.374                  0.498   233.29   
2006-12-16 17:27:00                5.388                  0.502   233.74   
2006-12-16 17:28:00                3.666                  0.528   235.68   

                     Sub_metering_1  Sub_metering_2  Sub_metering_3  
Datetime                                                             
2006-12-16 17:24:00             0.0             1.0            17.0  
2006-12-16 17:25:00             0.0             1.0            16.0  
2006-12-16 17:26:00             0.0             2.0            17.0  
2006-12-16 17:27:00             0.0             1.0            17.0  
2006-12-16 17:28:00         

In [20]:
scaler = MinMaxScaler()
scaled_data =scaler.fit_transform(data)
print("\nScaled Data Shape")
print(scaled_data.shape)


Scaled Data Shape
(1048575, 6)


In [40]:
sequence_length = 30

X= []
y = []

for i in range(sequence_length, len(scaled_data)):
    X.append(scaled_data[i-sequence_length:i])
    y.append(scaled_data[i, 0])

In [42]:
X=np.array(X)
y=np.array(y)
print("\n X shape:", X.shape)
print("\n y shape:", y.shape)


 X shape: (1048545, 30, 6)

 y shape: (1048545,)


In [43]:
train_size = int(len(X) * 0.8)
X_train = X[:train_size]
X_test = X[train_size:]
y_train = y[:train_size]
y_test =  y[train_size:]
print("\nTraining sample:", len(X_train))
print("\nTesting sample:", len(X_test))

print("\nX_train shape:", X_train.shape)
print("\nX_test shape:", X_test.shape)

print("\ny_train shape:", y_train.shape)
print("\ny_test shape:", y_test.shape)


Training sample: 838836

Testing sample: 209709

X_train shape: (838836, 30, 6)

X_test shape: (209709, 30, 6)

y_train shape: (838836,)

y_test shape: (209709,)


In [45]:
print("\nOne Sequence shape:")
print(X_train[0].shape)

print("\nTarget value:")
print(y_train[0])


One Sequence shape:
(30, 6)

Target value:
0.24957523126297906


In [46]:
model = Sequential()
model.add(
    SimpleRNN(
        units=64,
        activation='tanh',
        return_sequences=True,
        input_shape=(X_train.shape[1], X_train.shape[2])
    )
)

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [47]:
model.add(Dropout(0.2))

In [48]:
model.add(
    SimpleRNN(
        units=32,
        activation='tanh',
    )
)

In [50]:
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)
print("\nModel compiled successfully!")


Model compiled successfully!


In [58]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [61]:
history=model.fit(X_train,epochs=20,batch_size=64,validation_data=(X_test,y_test),callbacks=[early_stopping],verbose=1)

Epoch 1/20


ValueError: None values not supported.